# Imports

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

# Path

In [2]:
CURATED_DIR = "../../data/curated"

# Load Dataset

In [3]:
realtime_df = pd.read_parquet(
    f"{CURATED_DIR}/realtime_environmental_master.parquet"
)

print(realtime_df.shape)
realtime_df.head()

(2856, 23)


,datetime,station_id,station_name,district,latitude,longitude,temperature_2m,relative_humidity_2m,dew_point_2m,precipitation,...,wind_direction_10m,wind_gusts_10m,pm10,pm2_5,nitrogen_dioxide,sulphur_dioxide,ozone,carbon_monoxide,european_aqi,us_aqi
0,2026-06-13 00:00:00,WB001,Avidipta Housing Complex,Kolkata,22.493217,88.39831,28.150000,91.859573,26.700001,0.0,...,144.605118,15.119999,59.000000,51.700001,21.600000,8.4,49.0,358.0,97.073326,158.337708
1,2026-06-13 01:00:00,WB001,Avidipta Housing Complex,Kolkata,22.493217,88.39831,27.950001,92.390945,26.600000,0.0,...,162.801376,15.119999,59.599998,52.000000,21.200001,8.0,48.0,383.0,94.636673,156.734650
2,2026-06-13 02:00:00,WB001,Avidipta Housing Complex,Kolkata,22.493217,88.39831,27.549999,94.297012,26.549999,0.0,...,170.537750,14.400000,61.900002,53.799999,21.400000,7.9,48.0,431.0,92.816658,155.537277
3,2026-06-13 03:00:00,WB001,Avidipta Housing Complex,Kolkata,22.493217,88.39831,27.250000,95.123306,26.400000,0.0,...,183.813995,11.520000,66.400002,57.900002,23.600000,8.3,46.0,521.0,91.869995,154.914474
4,2026-06-13 04:00:00,WB001,Avidipta Housing Complex,Kolkata,22.493217,88.39831,27.100000,95.118042,26.250000,0.0,...,157.890503,11.520000,71.000000,61.700001,26.299999,8.9,44.0,634.0,91.433334,154.627197


# Dataset Overview

In [4]:
print("Rows:", realtime_df.shape[0])

print("Columns:", realtime_df.shape[1])

print(
    "Stations:",
    realtime_df["station_id"].nunique()
)

print(
    "Start:",
    realtime_df["datetime"].min()
)

print(
    "End:",
    realtime_df["datetime"].max()
)

Rows: 2856
Columns: 23
Stations: 17
Start: 2026-06-13 00:00:00
End: 2026-06-19 23:00:00


# Missing Value Analysis

In [5]:
missing_report = pd.DataFrame({

    "missing_count":
        realtime_df.isnull().sum(),

    "missing_pct":
        (
            realtime_df.isnull().mean()
            * 100
        ).round(2)

}).sort_values(
    "missing_pct",
    ascending=False
)

missing_report

,missing_count,missing_pct
datetime,0,0.0
station_id,0,0.0
station_name,0,0.0
district,0,0.0
latitude,0,0.0
longitude,0,0.0
temperature_2m,0,0.0
relative_humidity_2m,0,0.0
dew_point_2m,0,0.0
precipitation,0,0.0


## Validation :

In [6]:
total_missing = (
    realtime_df
    .isnull()
    .sum()
    .sum()
)

print(
    "Total Missing Values:",
    total_missing
)

Total Missing Values: 0


# Duplicate Analysis

In [7]:
duplicates = realtime_df.duplicated(
    subset=[
        "station_id",
        "datetime"
    ]
).sum()

print(
    "Duplicate Records:",
    duplicates
)

Duplicate Records: 0


# Station Coverage

In [8]:
station_coverage = (
    realtime_df
    .groupby(
        "station_id"
    )
    .size()
    .reset_index(
        name="records"
    )
    .sort_values(
        "station_id"
    )
)

station_coverage

,station_id,records
0,WB001,168
1,WB002,168
2,WB003,168
3,WB004,168
4,WB005,168
5,WB006,168
6,WB007,168
7,WB008,168
8,WB009,168
9,WB010,168


## Validation :

In [9]:
print(
    station_coverage["records"]
    .describe()
)

count     17.0
mean     168.0
std        0.0
min      168.0
25%      168.0
50%      168.0
75%      168.0
max      168.0
Name: records, dtype: float64


# Temporal Coverage
---

## Hourly continuity :

In [10]:
hours = realtime_df["datetime"].nunique()
print(
    "Unique Hours:",
    hours
)

Unique Hours: 168


## Hourly distribution :

In [11]:
hourly_counts = (
    realtime_df
    .groupby(
        "datetime"
    ).size()
)

hourly_counts.head()

datetime
2026-06-13 00:00:00    17
2026-06-13 01:00:00    17
2026-06-13 02:00:00    17
2026-06-13 03:00:00    17
2026-06-13 04:00:00    17
dtype: int64

In [12]:
hourly_counts.describe()

count    168.0
mean      17.0
std        0.0
min       17.0
25%       17.0
50%       17.0
75%       17.0
max       17.0
dtype: float64

# Environmental Range Validation
---

# Temparature

In [13]:
print(
    realtime_df["temperature_2m"]
    .describe()
)

count    2856.000000
mean       29.748949
std         2.424249
min        25.000000
25%        27.950001
50%        29.250000
75%        31.299999
max        36.200001
Name: temperature_2m, dtype: float64


# Humidity

In [14]:
print(

    realtime_df["relative_humidity_2m"]

    .describe()

)

count    2856.000000
mean       81.645493
std        11.314012
min        46.094902
25%        74.972780
50%        84.515739
75%        90.572655
max        98.240906
Name: relative_humidity_2m, dtype: float64


# AQI Pollutants

In [15]:
aqi_cols = [

    "pm10",
    "pm2_5",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone",
    "carbon_monoxide"

]

realtime_df[aqi_cols].describe()

,pm10,pm2_5,nitrogen_dioxide,sulphur_dioxide,ozone,carbon_monoxide
count,2856.000000,2856.000000,2856.000000,2856.000000,2856.000000,2856.000000
mean,58.005886,48.240196,12.627662,11.532528,102.502098,413.748962
std,16.590216,16.263475,8.355685,5.074420,41.871548,86.020508
min,32.700001,26.100000,1.800000,6.100000,24.000000,228.000000
25%,48.099998,38.799999,6.000000,8.100000,65.000000,347.000000
50%,55.099998,45.299999,11.500000,9.850000,95.000000,404.000000
75%,65.000000,52.299999,16.925000,13.600000,133.000000,473.000000
max,128.500000,124.300003,50.400002,36.099998,249.000000,722.000000


# Statistical Summary
---

In [16]:
realtime_df.describe().T

,count,mean,min,25%,50%,75%,max,std
datetime,2856,2026-06-16 11:30:00,2026-06-13 00:00:00,2026-06-14 17:45:00,2026-06-16 11:30:00,2026-06-18 05:15:00,2026-06-19 23:00:00,NaN
latitude,2856.0,22.550886,22.48127,22.511385,22.548628,22.576386,22.692911,0.053656
longitude,2856.0,88.386444,88.284554,88.361045,88.382256,88.40998,88.509293,0.050741
temperature_2m,2856.0,29.748949,25.0,27.950001,29.25,31.299999,36.200001,2.424249
relative_humidity_2m,2856.0,81.645493,46.094902,74.97278,84.515739,90.572655,98.240906,11.314012
dew_point_2m,2856.0,26.073004,22.1,25.5,26.15,26.75,28.6,0.958899
precipitation,2856.0,0.418662,0.0,0.0,0.0,0.1,19.1,1.670929
surface_pressure,2856.0,1002.922729,999.054138,1001.928802,1002.966339,1003.948532,1006.700012,1.411438
cloud_cover,2856.0,56.571777,1.0,23.0,54.0,95.0,100.0,34.406082
wind_speed_10m,2856.0,8.199567,0.509117,6.140478,8.311245,10.164418,20.750027,3.150761


In [19]:
summary_stats = realtime_df.describe().T

summary_stats.to_csv(
    "../../src/reports/realtime_summary_statistics.csv"
)

print("Saved: ../../src/reports/realtime_summary_statistics.csv")

Saved: ../../src/reports/realtime_summary_statistics.csv


# Quality Scorecard

In [17]:
quality_scorecard = pd.DataFrame({

    "Check": [
        "Missing Values",
        "Duplicate Records",
        "Station Coverage",
        "Temporal Coverage"
    ],

    "Result": [
        total_missing == 0,
        duplicates == 0,
        station_coverage["records"].min() == 168,
        hourly_counts.min() == 17
    ]
})

quality_scorecard

,Check,Result
0,Missing Values,True
1,Duplicate Records,True
2,Station Coverage,True
3,Temporal Coverage,True


# Export Report

In [18]:
quality_scorecard.to_csv(
    "../../src/reports/realtime_quality_report.csv",
    index=False
)

print(
    "Report Saved"
)

Report Saved
